In [1]:
import sys

try:
    from google.protobuf import message_factory

    if not hasattr(message_factory.MessageFactory, "GetPrototype"):

        def _GetPrototype(self, descriptor):
            return self.GetMessageClass(descriptor)

        message_factory.MessageFactory.GetPrototype = _GetPrototype
except Exception:
    pass

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

np.random.seed(42)
tf.random.set_seed(42)



2026-05-05 07:19:41.957157: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777965581.970683      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777965581.974804      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-05 07:19:41.991664: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
plt.rcParams["figure.figsize"] = (10, 10)



In [3]:
import os


def read_csv_safe(relative_path):
    possible_paths = [
        os.path.join("/kaggle/input/leaf-classification", relative_path),
        os.path.join("../input", relative_path),
        relative_path,
    ]
    for p in possible_paths:
        if os.path.exists(p):
            return pd.read_csv(p)
    raise FileNotFoundError(f"Unable to locate {relative_path}")


train_path = "train.csv"
data = read_csv_safe(train_path)
ids = data.pop("id")  # store ids, not used for training



In [4]:
# Prepare labels
y_raw = data.pop("species")
le = LabelEncoder()
y_int = le.fit_transform(y_raw)  # integer encoded labels
y_cat = to_categorical(y_int)  # one‑hot encoding

# Compute class weights to handle imbalance
class_weights_array = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_int), y=y_int
)
class_weight_dict = {i: w for i, w in enumerate(class_weights_array)}

# Standardise features
scaler = StandardScaler().fit(data.values)
X = scaler.transform(data.values)

# Use a larger validation split so that each of the ~99 classes appears at least once
X_train, X_val, y_train_cat, y_val_cat = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y_int
)



In [5]:
num_features = X.shape[1]  # should be 192
num_classes = y_cat.shape[1]  # should be 99

model = Sequential()
model.add(
    Dense(
        1024,
        input_dim=num_features,
        kernel_initializer="glorot_uniform",
        activation="relu",
    )
)
model.add(Dropout(0.3))  # increased dropout for better regularisation
model.add(Dense(512, kernel_initializer="glorot_normal", activation="relu"))
model.add(Dense(256, activation="relu"))
model.add(Dense(128, activation="relu"))
model.add(Dropout(0.3))  # increased dropout
model.add(Dense(64, activation="relu"))
model.add(Dense(num_classes, activation="softmax"))



/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-05-05 07:19:48.441011: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777965588.441434      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:06:00.0, compute capability: 8.6


In [6]:
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.0)
model.compile(
    loss=loss_fn,
    optimizer=Adam(learning_rate=1e-4),
    metrics=["accuracy"],
)



In [7]:
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "best_model.h5", save_best_only=True, monitor="val_loss", mode="min"
)

early_stop_cb = EarlyStopping(
    monitor="val_loss", patience=30, restore_best_weights=True
)

reduce_lr_cb = ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=8, min_lr=1e-6, verbose=0
)

history = model.fit(
    X_train,
    y_train_cat,
    batch_size=32,
    epochs=2000,  # allow more epochs; early stopping will halt when appropriate
    verbose=0,
    validation_data=(X_val, y_val_cat),
    callbacks=[checkpoint_cb, early_stop_cb, reduce_lr_cb],
    class_weight=class_weight_dict,
)



I0000 00:00:1777965590.076882     111 service.cc:148] XLA service 0x7fec90004640 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777965590.076910     111 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-05 07:19:50.102112: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777965590.233673     111 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-05-05 07:19:52.090422: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1200', 68 bytes spill stores, 72 bytes spill loads

2026-05-05 07:19:52.635768: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1202', 24 bytes spill 

In [8]:
print("Best val accuracy:", max(history.history["val_accuracy"]))



Best val accuracy: 0.9832402467727661


In [9]:
test_path = "test.csv"
test_df = read_csv_safe(test_path)
test_ids = test_df.pop("id")
X_test = scaler.transform(test_df.values)



In [10]:
model.load_weights("best_model.h5")  # use the best validation weights
y_pred_probs = model.predict(X_test, verbose=0)  # shape (n_test, n_classes)

# Clip probabilities to the required range
y_pred_probs = np.clip(y_pred_probs, 1e-15, 1 - 1e-15)



2026-05-05 07:20:22.051830: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 220 bytes spill stores, 220 bytes spill loads

2026-05-05 07:20:22.592145: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 4 bytes spill stores, 4 bytes spill loads



In [11]:
sample_sub_path = "sample_submission.csv"
sample_sub = read_csv_safe(sample_sub_path)

class_cols = sample_sub.columns.tolist()[1:]  # all species columns

# Map original class names to the indices produced by LabelEncoder
class_to_idx = {cls: idx for idx, cls in enumerate(le.classes_)}
ordered_preds = np.zeros((y_pred_probs.shape[0], len(class_cols)))

for col_idx, cls_name in enumerate(class_cols):
    if cls_name in class_to_idx:
        ordered_preds[:, col_idx] = y_pred_probs[:, class_to_idx[cls_name]]
    else:
        ordered_preds[:, col_idx] = 1e-15  # fallback for missing classes

submission = pd.DataFrame(ordered_preds, columns=class_cols)
submission.insert(0, "id", test_ids.values)



In [12]:
submission_path = "submission_nn_kernel.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")

Submission written to submission_nn_kernel.csv
